# Task 04 — SASRec via RePlay

Run All notebook for a clean Google Colab GPU runtime. It clones the project, installs the pinned SASRec extra, accepts the four original competition CSV files when they are absent, and runs the same leakage-safe chronological 90/10 pipeline used locally.

In [ ]:
import subprocess

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('GPU is unavailable. In Colab select Runtime > Change runtime type > T4 GPU, then Run All again.')
print('GPU:', gpu.stdout.strip())

## Reproducible project setup

`PROJECT_REF` is explicit so a commit hash or release tag can replace `main` for an immutable rerun. RePlay is pinned by the project's `sasrec` optional dependency.

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_REPO_URL = 'https://github.com/sblenlkj/hse_rec_system.git'
PROJECT_REF = 'main'  # Prefer the exact Task 04 commit hash for archival reruns.
PROJECT_DIR = Path('/content/hse_rec_system')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', PROJECT_REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', PROJECT_REF], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', f'{PROJECT_DIR}[sasrec]'], check=True)
os.chdir(PROJECT_DIR)

import replay, torch, lightning
print({'replay': replay.__version__, 'torch': torch.__version__, 'lightning': lightning.__version__, 'cuda': torch.cuda.get_device_name(0)})

## Competition CSV upload

Upload the original `events.csv`, `item_features.csv`, `user_features.csv`, and `submission_sample.csv` together when prompted. The notebook performs all deterministic conversion itself; competition data is never added to git.

In [ ]:
from google.colab import files

required = ['events.csv', 'item_features.csv', 'user_features.csv', 'submission_sample.csv']
data_dir = PROJECT_DIR / 'data'
data_dir.mkdir(exist_ok=True)
missing = [name for name in required if not (data_dir / name).exists()]
if missing:
    print('Upload these files:', missing)
    uploaded = files.upload()
    for name in missing:
        if name not in uploaded:
            raise FileNotFoundError(f'Missing uploaded file: {name}')
        (data_dir / name).write_bytes(uploaded[name])
print({name: (data_dir / name).stat().st_size for name in required})

## Full validation, checkpoint, top-200 candidates, full-data refit, submission

This single command owns the unchanged chronological split and train-only assertions. It writes validation/full checkpoints, scored and ranked candidates, metrics JSON, submission #4, and a downloadable zip.

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_DIR / 'src')
subprocess.run([
    'python', '-m', 'recsys.sasrec_pipeline',
    '--mode', 'full', '--device', 'cuda',
    '--data-dir', str(data_dir),
], check=True, env=env)

In [ ]:
import json
import pandas as pd

result_path = PROJECT_DIR / '.artifacts/004_sasrec_validation.json'
result = json.loads(result_path.read_text())
display(result)
candidates = pd.read_csv(PROJECT_DIR / '.artifacts/candidates/4_sasrec_full_top200.csv')
assert candidates.groupby('user_id').size().eq(200).all()
assert list(candidates.columns) == ['user_id', 'item_id', 'sasrec_score', 'sasrec_rank']
print('Validated full-data candidate rows:', len(candidates))

## Download artifacts

The zip preserves `.artifacts/...` paths so it can be extracted directly into a local project checkout without manual conversion.

In [ ]:
bundle = PROJECT_DIR / '.artifacts/4_sasrec_artifacts.zip'
print('Bundle:', bundle, 'bytes:', bundle.stat().st_size)
files.download(str(bundle))